# MobileNetV3-Small — CORREGIDO

**Fix aplicado:** se quitó `include_preprocessing=False`, que dejaba las imágenes sin normalizar y arruinaba el entrenamiento (F1 ≈ 0.49).

Con el modelo en su modo por defecto, la normalización ocurre dentro de la red y recibe las imágenes en rango [0–255], igual que EfficientNetB0.


## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install scikit-learn scipy -q

import os, gc, json, shutil, warnings
import numpy as np
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.utils import shuffle
from sklearn.metrics import (f1_score, precision_score,
                              recall_score, cohen_kappa_score)
import pandas as pd
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

Mounted at /content/drive
TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Dataset (1000 imágenes por clase)

In [2]:
DATASET_PATH       = '/content/drive/MyDrive/tesis_palta/dataset_balanceado'
RESULTS_DIR        = '/content/drive/MyDrive/tesis_palta/resultados_OE1'
IMG_SIZE           = (224, 224)
BATCH              = 8
K                  = 10
NUM_CLASES         = 3
SEED               = 42
IMAGENES_POR_CLASE = 1000

os.makedirs(RESULTS_DIR, exist_ok=True)

clases_carpetas = sorted(os.listdir(DATASET_PATH))
print(f"Clases (orden del modelo): {clases_carpetas}")

rutas_todas, etiquetas_todas = [], []
for idx, clase in enumerate(clases_carpetas):
    carpeta = os.path.join(DATASET_PATH, clase)
    for archivo in os.listdir(carpeta):
        if archivo.lower().endswith(('.jpg','.jpeg','.png')):
            rutas_todas.append(os.path.join(carpeta, archivo))
            etiquetas_todas.append(idx)

rutas_todas     = np.array(rutas_todas)
etiquetas_todas = np.array(etiquetas_todas)

np.random.seed(SEED)
indices = []
for clase_idx in range(NUM_CLASES):
    idx_clase = np.where(etiquetas_todas == clase_idx)[0]
    selec = np.random.choice(
        idx_clase,
        size=min(IMAGENES_POR_CLASE, len(idx_clase)),
        replace=False)
    indices.extend(selec)

rutas     = rutas_todas[np.array(indices)]
etiquetas = etiquetas_todas[np.array(indices)]
rutas, etiquetas = shuffle(rutas, etiquetas, random_state=SEED)

print(f"Total: {len(rutas)} imágenes")
print(f"Distribución: {np.bincount(etiquetas)}")

Clases (orden del modelo): ['Antracnosis', 'Roña', 'Sano']
Total: 3000 imágenes
Distribución: [1000 1000 1000]


## Preprocesamiento

`preprocess_input` de MobileNetV3 es pasa-todo (no hace nada): la normalización la hace ahora el modelo internamente. Las imágenes entran en [0–255].

In [3]:
def crear_dataset(rutas_fold, etiquetas_fold, entrenamiento=True):
    def cargar_imagen(ruta, etiqueta):
        img = tf.io.read_file(ruta)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, IMG_SIZE)
        img = tf.cast(img, tf.float32)
        img = tf.keras.applications.mobilenet_v3.preprocess_input(img)  # pasa-todo
        return img, etiqueta

    def aumentar(img, etiqueta):
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, 0.1)
        img = tf.image.random_contrast(img, 0.9, 1.1)
        return img, etiqueta

    ds = tf.data.Dataset.from_tensor_slices((rutas_fold, etiquetas_fold))
    ds = ds.map(cargar_imagen, num_parallel_calls=tf.data.AUTOTUNE)
    if entrenamiento:
        ds = ds.map(aumentar, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.shuffle(300, seed=SEED)
    ds = ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)
    return ds

print("✅ Preprocesamiento listo")

✅ Preprocesamiento listo


## MobileNetV3-Small (FIX APLICADO)

In [4]:
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

def construir_modelo_mv3():
    tf.keras.backend.clear_session()
    gc.collect()

    # FIX: se eliminó include_preprocessing=False.
    # El modelo queda en su modo por defecto (include_preprocessing=True),
    # normaliza internamente y espera imágenes en [0-255].
    base = tf.keras.applications.MobileNetV3Small(
        input_shape=(224, 224, 3),
        include_top=False,
        weights='imagenet'
    )
    base.trainable = False

    entrada = tf.keras.Input(shape=(224, 224, 3))
    x = base(entrada, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    salida = layers.Dense(NUM_CLASES, activation='softmax')(x)

    modelo = models.Model(entrada, salida)
    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return modelo, base

def get_callbacks():
    return [
        EarlyStopping('val_accuracy', patience=6,
                      restore_best_weights=True, verbose=0),
        ReduceLROnPlateau('val_loss', factor=0.5,
                          patience=3, min_lr=1e-7, verbose=0)
    ]

print("✅ MobileNetV3-Small corregido listo")

✅ MobileNetV3-Small corregido listo


## Entrenamiento 10-fold (con re-inicio seguro)

`REENTRENAR_DESDE_CERO = True` respalda tus resultados viejos (con bug) como `_BUGGY.json` y arranca limpio. Si Colab se desconecta a mitad, puedes ponerlo en `False` para retomar donde quedó.

In [6]:
REENTRENAR_DESDE_CERO = True   # True = arranca limpio con el fix

ruta_json = f'{RESULTS_DIR}/resultados_MobileNetV3Small.json'

# Respaldar resultados anteriores (con el bug) y reiniciar
if REENTRENAR_DESDE_CERO and os.path.exists(ruta_json):
    shutil.copy(ruta_json, ruta_json.replace('.json', '_BUGGY.json'))
    os.remove(ruta_json)
    print("♻️  Resultados anteriores respaldados como *_BUGGY.json y reiniciados")

if os.path.exists(ruta_json):
    with open(ruta_json, 'r') as f:
        resultados_fold = json.load(f)
    print(f"⚡ Retomando desde fold {len(resultados_fold)+1}/10")
else:
    resultados_fold = []
    print("🆕 Iniciando desde cero")

print(f"\n{'='*50}")
print(f"  ENTRENANDO: MobileNetV3Small")
print(f"{'='*50}")

kfold = StratifiedKFold(n_splits=K, shuffle=True, random_state=SEED)

for fold_idx, (idx_train, idx_val) in enumerate(kfold.split(rutas, etiquetas)):
    fold_num = fold_idx + 1

    if fold_num <= len(resultados_fold):
        print(f"  Fold {fold_num}/10 → ya guardado ✅")
        continue

    print(f"\n  Fold {fold_num}/10 ", end="", flush=True)

    ds_train = crear_dataset(rutas[idx_train], etiquetas[idx_train], True)
    ds_val   = crear_dataset(rutas[idx_val],   etiquetas[idx_val],   False)

    modelo, base = construir_modelo_mv3()

    # FASE 1 — base congelada
    print("fase1..", end="", flush=True)
    modelo.fit(ds_train, validation_data=ds_val,
               epochs=15, callbacks=get_callbacks(), verbose=0)

    # FASE 2 — fine-tuning últimas 15 capas
    print("fase2..", end="", flush=True)
    base.trainable = True
    for capa in base.layers[:-15]:
        capa.trainable = False

    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(1e-5),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    modelo.fit(ds_train, validation_data=ds_val,
               epochs=15, callbacks=get_callbacks(), verbose=0)

    # Predicciones
    y_pred, y_true = [], []
    for imgs, labs in ds_val:
        y_pred.extend(np.argmax(modelo.predict(imgs, verbose=0), axis=1))
        y_true.extend(labs.numpy())

    y_pred = np.array(y_pred)
    y_true = np.array(y_true)

    p  = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    r  = recall_score(y_true, y_pred,    average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred,        average='weighted', zero_division=0)
    kp = cohen_kappa_score(y_true, y_pred)

    resultados_fold.append({
        'Fold': fold_num,
        'Precision': round(p, 4),
        'Recall':    round(r, 4),
        'F1':        round(f1, 4),
        'Kappa':     round(kp, 4)
    })

    with open(ruta_json, 'w') as f:
        json.dump(resultados_fold, f)

    print(f"→ F1:{f1:.4f} K:{kp:.4f} ✅", flush=True)

    del modelo, base, ds_train, ds_val
    gc.collect()
    tf.keras.backend.clear_session()

df = pd.DataFrame(resultados_fold)
print(f"\n✅ MobileNetV3Small (corregido) — {len(resultados_fold)}/10 folds")
print(f"   F1:    {df['F1'].mean():.4f} ± {df['F1'].std():.4f}")
print(f"   Kappa: {df['Kappa'].mean():.4f} ± {df['Kappa'].std():.4f}")

🆕 Iniciando desde cero

  ENTRENANDO: MobileNetV3Small

  Fold 1/10 fase1..fase2..→ F1:0.8711 K:0.8050 ✅

  Fold 2/10 fase1..fase2..→ F1:0.9031 K:0.8550 ✅

  Fold 3/10 fase1..fase2..→ F1:0.8968 K:0.8450 ✅

  Fold 4/10 fase1..fase2..→ F1:0.8968 K:0.8450 ✅

  Fold 5/10 fase1..fase2..→ F1:0.9035 K:0.8550 ✅

  Fold 6/10 fase1..fase2..→ F1:0.9037 K:0.8550 ✅

  Fold 7/10 fase1..fase2..→ F1:0.8710 K:0.8050 ✅

  Fold 8/10 fase1..fase2..→ F1:0.8648 K:0.7950 ✅

  Fold 9/10 fase1..fase2..→ F1:0.8844 K:0.8250 ✅

  Fold 10/10 fase1..fase2..→ F1:0.8868 K:0.8300 ✅

✅ MobileNetV3Small (corregido) — 10/10 folds
   F1:    0.8882 ± 0.0149
   Kappa: 0.8315 ± 0.0231
